# Education 360° — Marketing & Lead Analysis
Analyze lead outcomes by source, branch, and generated month. Marketing campaign aggregates are treated separately from the individual lead table.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["branches_dim.csv", "braches_dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses_dim.csv"],
    "date table": ["DateTable.csv", "date_table.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students_dim.csv"]
}

def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

In [ ]:
leads = datasets["leads"].copy()
marketing = datasets["marketing"].copy()
if "generated date" in leads:
    leads["generated date"] = pd.to_datetime(leads["generated date"], errors="coerce")
display(leads.head())
display(marketing.head())

## Lead status

In [ ]:
status_summary = leads["lead status"].value_counts(dropna=False).rename_axis("Lead Status").reset_index(name="Leads")
display(status_summary)
converted = (leads["lead status"].astype(str).str.strip().str.lower() == "converted").sum()
conversion_rate = converted / len(leads) * 100 if len(leads) else 0
print(f"Converted leads: {converted:,}")
print(f"Lead conversion rate: {conversion_rate:.2f}%")

## Lead source performance

In [ ]:
source_summary = leads.groupby("lead source").agg(
    Total_Leads=("lead id","nunique"),
    Converted_Leads=("lead status", lambda s: (s.astype(str).str.strip().str.lower()=="converted").sum())
).reset_index()
source_summary["Conversion_Rate_%"] = (
    source_summary["Converted_Leads"] / source_summary["Total_Leads"] * 100
)
display(source_summary.sort_values("Conversion_Rate_%", ascending=False))
source_summary.plot(x="lead source", y="Total_Leads", kind="bar", figsize=(9,4),
                    title="Leads by Source", legend=False)
plt.xticks(rotation=35); plt.tight_layout(); plt.show()

## Branch performance

In [ ]:
branch_summary = leads.groupby("branch id").agg(
    Leads=("lead id","nunique"),
    Converted=("lead status", lambda s: (s.astype(str).str.strip().str.lower()=="converted").sum())
).reset_index()
branch_summary["Conversion Rate %"] = branch_summary["Converted"] / branch_summary["Leads"] * 100
display(branch_summary)

## Marketing campaign overview

In [ ]:
display(marketing.describe(include="all").T)
numeric = marketing.select_dtypes(include="number")
display(numeric.sum().to_frame("Total").join(numeric.mean().to_frame("Average")))

## Interpretation caution
The campaign-level Marketing table has a different row count/grain from Leads. Do not combine campaign leads with individual lead records unless a valid campaign key and matching grain are confirmed.